# Theseus 教程（中文翻译版）

- 原始英文版：`01_least_squares_optimization.ipynb`
- 说明：本文件为自动翻译版本（保留代码不翻译，清空输出以减小体积）。如遇术语不一致，可优先参考英文原文。


<h1>使用Theseus进行最小二乘优化</h1>
本教程演示如何使用Theseus 解决曲线拟合问题。本教程中的示例受到 [Ceres](https://ceres-solver.org/) [教程](http://ceres-solver.org/nnls_tutorial.html) 的启发，其结构类似于 Ceres 中的[曲线拟合示例](http://ceres-solver.org/nnls_tutorial.html#curve-fitting) 和[稳健曲线拟合示例](http://ceres-solver.org/nnls_tutorial.html#robust-curve-fitting)。

<h2>二次曲线</h2>
在本教程中，我们将展示如何拟合二次函数：<i>y = ax<sup>2</sup> + b</i>

<h3>第0步：生成数据</h3>
我们首先通过二次函数 <i>x<sup>2</sup> + 0.5</i> 的采样点生成数据。为此，我们添加高斯噪声 <i>&sigma; = 0.01。

In [ ]:
import torch

torch.manual_seed(0)

def generate_data(num_points=100, a=1, b=0.5, noise_factor=0.01):
    # Generate data: 100 points sampled from the quadratic curve listed above
    data_x = torch.rand((1, num_points))
    noise = torch.randn((1, num_points)) * noise_factor
    data_y = a * data_x.square() + b + noise
    return data_x, data_y

data_x, data_y = generate_data()

# Plot the data
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.scatter(data_x, data_y);
ax.set_xlabel('x');
ax.set_ylabel('y');

我们通过 3 个步骤演示如何使用Theseus 来解决这个曲线拟合问题：
<ul>
<li>第 1 步：表示数据和变量
<li>第 2 步：设置优化
<li>第 3 步：运行优化
</ul>

<h3>步骤1：在Theseus中表示数据和变量</h3>
正如我们在教程 0 中所述，Theseus 变量在语义上分为两个主要类别：
<ul>
<li><i>优化变量</i>：那些将被我们的非线性最小二乘优化器修改以最小化总成本函数的变量
<li><i>辅助变量</i>：成本函数执行优化所需的其他变量，但非线性最小二乘优化器不会对其进行优化，例如本例中的应用程序数据（我们将看到更多示例）
</ul>

我们的第一步是在Theseus 数据结构中表示数据 <i>(x, y)</i> 和优化变量（<i>a</i> 和 <i>b</i>）。
优化变量的类型必须是 `Manifold`。对于此示例，我们选择其 `Vector` 子类来表示 <i>a</i> 和 <i>b</i>。因为它们是一维量，所以我们在初始化这些 `Vector` 对象时只需要 1 个自由度。 （或者，我们也可以将这两个变量表示为单个二维 `Vector` 对象；但是，这会改变错误函数的编写方式。）（辅助）数据变量可以是任何 `Variable` 类型的实例。对于此示例，类型 `Variable` 本身就足够了。

In [ ]:
import theseus as th

# data is of type Variable
x = th.Variable(data_x, name="x")
y = th.Variable(data_y, name="y")

# optimization variables are of type Vector with 1 degree of freedom (dof)
a = th.Vector(1, name="a")
b = th.Vector(1, name="b")

<h3>第2步：设置优化</h3>
    
最小二乘拟合的残余误差在 `CostFunction` 中捕获。在本例中，我们将使用Theseus 提供的 `AutoDiffCostFunction`，它提供了一种易于使用的方法来捕获任意成本函数。 `AutoDiffCostFunction`只需要我们定义优化变量和辅助变量，并提供计算残差的误差函数。从那里，它使用 PyTorch autograd 通过自动微分计算优化变量的雅可比行列式。

在下面的示例中，`quad_error_fn` 捕获与两个一维 `Vector` 对象 `a`、`b` 拟合的二次函数的最小二乘误差。

总最小二乘误差可以通过一个 100 维 `AutoDiffCostFunction`（其中每个维度代表一个数据点的误差）或一组 100 个一维 `AutoDiffCostFunction`（其中每个成本函数捕获一个数据点的误差）来捕获。我们在此示例中使用前者（即一个 100 维 `AutoDiffCostFunction`），但我们将在教程 4 和 5 中看到后者的示例。

最后，我们将成本函数组合成Theseus 优化问题：
- 优化标准由`Objective`表示。这是通过将所有成本函数添加到其中来构造的。
- 然后我们可以选择一个优化器并设置其一些默认配置（例如，下面示例中的 `GaussNewton` 和 `max_iterations=15`）。
- 然后使用目标及其关联的优化器来构造 `TheseusLayer`，它代表一层优化

In [ ]:
def quad_error_fn(optim_vars, aux_vars):
    a, b = optim_vars 
    x, y = aux_vars
    est = a.tensor * x.tensor.square() + b.tensor
    err = y.tensor - est
    return err

optim_vars = a, b
aux_vars = x, y
cost_function = th.AutoDiffCostFunction(
    optim_vars, quad_error_fn, 100, aux_vars=aux_vars, name="quadratic_cost_fn"
)
objective = th.Objective()
objective.add(cost_function)
optimizer = th.GaussNewton(
    objective,
    max_iterations=15,
    step_size=0.5,
)
theseus_optim = th.TheseusLayer(optimizer)

<h3>第3步：运行优化</h3> 
现在运行优化问题只需要我们提供输入数据和初始值，并调用 `TheseusLayer` 上的前向函数。

输入以字典的形式提供，其中键代表优化变量（与其初始值配对）或辅助变量（与其数据配对）。字典 `theseus_inputs` 显示了一个示例。

有了这个输入，我们现在可以在Theseus 中运行最小二乘优化。我们通过调用 `TheseusLayer` 上的 `forward` 函数来完成此操作。每次调用 `forward` 函数后都会返回两个数量：
1. `updated_inputs` 对象，保存优化变量的最终值以及未更改的辅助变量值。这允许我们使用 `updated_inputs` 作为下游函数或Theseus 层的输入（例如，对于需要多次前向传递的问题，正如我们将在教程 2 中看到的那样。）
2. `info` 对象，必要时可以跟踪最佳解决方案，并保存有关优化的其他有用信息。跟踪最佳解决方案非常有用，因为如果误差从早期迭代开始增加，优化算法不会停止。 （执行反向传播时，最佳解决方案并不那么有用，因为反向传播使用整个优化序列；请参阅教程 2。）

In [ ]:
theseus_inputs = {
"x": data_x,
"y": data_y,
"a": 2 * torch.ones((1, 1)),
"b": torch.ones((1, 1))
}
with torch.no_grad():
    updated_inputs, info = theseus_optim.forward(
        theseus_inputs, optimizer_kwargs={"track_best_solution": True, "verbose":True})
print("Best solution:", info.best_solution)

# Plot the optimized function
fig, ax = plt.subplots()
ax.scatter(data_x, data_y);

a = info.best_solution['a'].squeeze()
b = info.best_solution['b'].squeeze()
x = torch.linspace(0., 1., steps=100)
y = a*x*x + b
ax.plot(x, y, color='k', lw=4, linestyle='--',
        label='Optimized quadratic')
ax.legend()

ax.set_xlabel('x');
ax.set_ylabel('y');

我们观察到，我们几乎完全恢复了采样的二次函数中使用的原始 <i>a, b</i> 值。

<h2>鲁二次棒圆形曲面</h2>

该示例还可以适用于误差被加权的问题，例如，使用柯西损失来减少具有极高误差的数据点的权重。这类似于 Ceres 解算器中的[稳健曲线拟合示例](http://ceres-solver.org/nnls_tutorial.html#robust-curve-fitting)。

在本教程中，我们进行简单的修改，将柯西损失权重添加到误差函数中：我们通过创建以下对其进行加权的 `cauchy_loss_quad_error_fn` 来替换上面 `AutoDiffCostFunction` 中的 `quad_error_fn`。

In [ ]:
def cauchy_fn(x):
    return torch.sqrt(0.5 * torch.log(1 + x ** 2))

def cauchy_loss_quad_error_fn(optim_vars, aux_vars):
    err = quad_error_fn(optim_vars, aux_vars)
    return cauchy_fn(err)

wt_cost_function = th.AutoDiffCostFunction(
    optim_vars, cauchy_loss_quad_error_fn, 100, aux_vars=aux_vars, name="cauchy_quad_cost_fn"
)

与上面的示例类似，我们现在可以使用此加权成本函数构建Theseus 优化问题：创建 `Objective`、优化器和 `TheseusLayer`，并运行优化。

In [ ]:
objective = th.Objective()
objective.add(wt_cost_function)
optimizer = th.GaussNewton(
    objective,
    max_iterations=20,
    step_size=0.3,
)
theseus_optim = th.TheseusLayer(optimizer)
theseus_inputs = {
"x": data_x,
"y": data_y,
"a": 2 * torch.ones((1, 1)),
"b": torch.ones((1, 1))
}

# We suppress warnings in this optimization call, because we observed that with this data, Cauchy 
# loss often results in singular systems with numerical computations as it approaches optimality. 
# Please note: getting a singular system during the forward optimization will throw
# an error if torch's gradient tracking is enabled.
import warnings
warnings.simplefilter("ignore")   

with torch.no_grad():
    _, info = theseus_optim.forward(
        theseus_inputs, optimizer_kwargs={"track_best_solution": True, "verbose":True})
print("Best solution:", info.best_solution)

# Plot the optimized function
fig, ax = plt.subplots()
ax.scatter(data_x, data_y);

a = info.best_solution['a'].squeeze()
b = info.best_solution['b'].squeeze()
x = torch.linspace(0., 1., steps=100)
y = a*x*x + b
ax.plot(x, y, color='k', lw=4, linestyle='--',
        label='Optimized quadratic')
ax.legend()

ax.set_xlabel('x');
ax.set_ylabel('y');

为了更有效地解决曲线拟合问题，还可以编写具有封闭形式雅可比行列式的自定义 `CostFunction`，而不是使用具有 torch 数值计算雅可比行列式的 `AutoDiffCostFunction`。我们在教程 3 中展示了一个例子。